In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
app_train = pd.read_csv('application_train.csv')
print(app_train.shape)
app_train.head()

In [ ]:
col_desc = pd.read_csv('HomeCredit_columns_description.csv', encoding='latin1')
col_desc[col_desc['Table'] == 'application_{train|test}.csv'][['Row', 'Description']]

In [ ]:
col_desc.head()

In [ ]:
app_train['TARGET'].value_counts()

In [ ]:
app_train['TARGET'].value_counts(normalize=True)


In [ ]:
sns.countplot(x='TARGET', data=app_train)
plt.title('Loan Repayment Status (0 = Repaid, 1 = Default)')
plt.show()

In [ ]:
app_train.isnull().sum()

In [ ]:
missing = app_train.isnull().sum() / len(app_train) * 100
missing

In [ ]:
missing = missing[missing > 0].sort_values(ascending=False)
missing.head(20)

In [ ]:
# Example: does income affect default rate?
app_train.groupby(pd.qcut(app_train['AMT_INCOME_TOTAL'], 10))['TARGET'].mean()

# Age (DAYS_BIRTH is negative, convert to years)
app_train['AGE_YEARS'] = -app_train['DAYS_BIRTH'] / 365
sns.histplot(data=app_train, x='AGE_YEARS', hue='TARGET', stat='density', common_norm=False)
plt.title('Age Distribution by Default Status')
plt.show()

In [ ]:
categorical_cols =app_train.select_dtypes(include='object').columns
print(categorical_cols)

# For low-cardinality columns, one-hot encode
app_train_encoded = pd.get_dummies(app_train, columns=categorical_cols, dummy_na=True)
print(app_train_encoded.shape)


In [ ]:
from sklearn.impute import SimpleImputer

# Separate target and ID first
X = app_train_encoded.drop(columns=['TARGET', 'SK_ID_CURR'])
y = app_train_encoded['TARGET']

imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

# Baseline: Decision Tree
dt = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict_proba(X_test)[:, 1]
print("Decision Tree AUC:", roc_auc_score(y_test, dt_pred))

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict_proba(X_test)[:, 1]
print("Random Forest AUC:", roc_auc_score(y_test, rf_pred))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

rf_pred_class = rf.predict(X_test)  # uses 0.5 threshold by default
print(classification_report(y_test, rf_pred_class))
print(confusion_matrix(y_test, rf_pred_class))

In [ ]:
cm=confusion_matrix(y_test, rf_pred_class)
print("Confusion Matrix:\n")
print(cm)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis')
plt.xlabel("predicted")
plt.ylabel("Actual")
plt.title("confusion Matrix")
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(15).plot(kind='barh')
plt.title('Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(rf)
X_test_sample = X_test.iloc[:100]
shap_values = explainer.shap_values(X_test_sample)

In [ ]:
print(type(shap_values))
if isinstance(shap_values, list):
    print(len(shap_values))
    print(shap_values[0].shape)
else:
    print(shap_values.shape)

In [ ]:
shap_values_class1 = shap_values[:, :, 1]
print(shap_values_class1.shape)  # should be (100, 261)

In [ ]:
shap.summary_plot(shap_values_class1, X_test_sample)

In [ ]:
applicant_idx = 0
applicant_shap = shap_values_class1[applicant_idx]

# Pair each feature with its SHAP value, sort by absolute impact
shap_df = pd.DataFrame({
    'feature': X_test_sample.columns,
    'shap_value': applicant_shap,
    'feature_value': X_test_sample.iloc[applicant_idx].values
})
shap_df['abs_impact'] = shap_df['shap_value'].abs()
shap_df = shap_df.sort_values('abs_impact', ascending=False)

print(shap_df.head(10))

In [ ]:
def get_applicant_shap_explanation(idx, X_sample, shap_values_class1, top_n=5):
    applicant_shap = shap_values_class1[idx]
    shap_df = pd.DataFrame({
        'feature': X_sample.columns,
        'shap_value': applicant_shap,
        'feature_value': X_sample.iloc[idx].values
    })
    shap_df['abs_impact'] = shap_df['shap_value'].abs()
    shap_df = shap_df.sort_values('abs_impact', ascending=False).head(top_n)
    
    # Convert to a clean list of dicts - easy to feed into a prompt later
    factors = []
    for _, row in shap_df.iterrows():
        direction = "increases risk" if row['shap_value'] > 0 else "decreases risk"
        factors.append({
            "feature": row['feature'],
            "value": row['feature_value'],
            "effect": direction,
            "impact_score": round(row['abs_impact'], 4)
        })
    return factors

# Test it
explanation = get_applicant_shap_explanation(0, X_test_sample, shap_values_class1)
for f in explanation:
    print(f)

In [ ]:
import os
print(os.getcwd())

In [ ]:
print(os.path.abspath('data/application_train.csv'))

In [ ]:
import os
print(os.path.exists('.env'))   # should print True

In [ ]:
!pip install google-genai

In [ ]:
from dotenv import load_dotenv
import os
from google import genai

load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

client = genai.Client(api_key=api_key)
print("Client created successfully")

In [ ]:
def generate_ai_explanation(prediction, probability, factors, audience="loan officer"):
    if client is None:
        return ("Gemini API key not found. Add GOOGLE_API_KEY to your .env file "
                "to enable AI-generated explanations.")

    factors_text = "\n".join([
        f"- {f['feature']} (value: {f['value']}): {f['effect']} (impact score: {f['impact_score']})"
        for f in factors
    ])

    prompt = f"""..."""  # keep your existing prompt here

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        return response.text
    except Exception as e:
        if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
            return ("⚠️ The Gemini API free-tier quota has been reached for now. "
                    "Please wait a minute and try again, or check your quota at "
                    "https://ai.google.dev/gemini-api/docs/rate-limits")
        return f"⚠️ An error occurred while generating the explanation: {e}"

In [ ]:
applicant_idx = 0
prediction = rf.predict(X_test_sample.iloc[[applicant_idx]])[0]
probability = rf.predict_proba(X_test_sample.iloc[[applicant_idx]])[0][1]

factors = get_applicant_shap_explanation(applicant_idx, X_test_sample, shap_values_class1)

explanation_officer = generate_ai_explanation(prediction, probability, factors, audience="loan officer")
print("LOAN OFFICER VERSION:\n", explanation_officer)

In [ ]:
explanation_applicant = generate_ai_explanation(prediction, probability, factors, audience="applicant")
print("\nAPPLICANT VERSION:\n", explanation_applicant)

In [ ]:
# Get predicted probabilities for everyone in the sample
probs = rf.predict_proba(X_test_sample)[:, 1]

# Find indices for a range of risk levels
low_risk_idx = probs.argmin()      # safest applicant in the sample
high_risk_idx = probs.argmax()     # riskiest applicant in the sample

# Also grab one "borderline" case near 50%
borderline_idx = (abs(probs - 0.5)).argmin()

test_cases = {
    "LOW RISK": low_risk_idx,
    "BORDERLINE": borderline_idx,
    "HIGH RISK": high_risk_idx
}

print("Selected applicant indices:", test_cases)
print("Their probabilities:", {k: round(probs[v], 4) for k, v in test_cases.items()})

In [ ]:
for label, idx in test_cases.items():
    prediction = rf.predict(X_test_sample.iloc[[idx]])[0]
    probability = rf.predict_proba(X_test_sample.iloc[[idx]])[0][1]
    factors = get_applicant_shap_explanation(idx, X_test_sample, shap_values_class1)
    
    explanation = generate_ai_explanation(prediction, probability, factors, audience="loan officer")
    
    print(f"\n{'='*60}")
    print(f"{label} — Applicant #{idx} (probability of default: {probability:.2%})")
    print('='*60)
    print(explanation)

In [ ]:
high_idx = test_cases["HIGH RISK"]
prediction = rf.predict(X_test_sample.iloc[[high_idx]])[0]
probability = rf.predict_proba(X_test_sample.iloc[[high_idx]])[0][1]
factors = get_applicant_shap_explanation(high_idx, X_test_sample, shap_values_class1)

explanation_applicant = generate_ai_explanation(prediction, probability, factors, audience="applicant")
print(explanation_applicant)

In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)

joblib.dump(rf, 'models/credit_risk_rf.pkl')
joblib.dump(imputer, 'models/imputer.pkl')
joblib.dump(list(X.columns), 'models/feature_columns.pkl')

# Save the sample data + shap values so the app doesn't need to recompute SHAP live
X_test_sample.to_pickle('models/X_test_sample.pkl')
import numpy as np
np.save('models/shap_values_class1.npy', shap_values_class1)

print("All artifacts saved successfully")

In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
print(os.path.exists('app.py'))
print(os.path.getsize('app.py'), "bytes")

In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)

joblib.dump(rf, 'models/credit_risk_rf.pkl')
joblib.dump(imputer, 'models/imputer.pkl')
joblib.dump(list(X.columns), 'models/feature_columns.pkl')

X_test_sample.to_pickle('models/X_test_sample.pkl')
import numpy as np
np.save('models/shap_values_class1.npy', shap_values_class1)

print("All artifacts saved successfully")

In [ ]:
print(os.listdir('models'))

In [ ]:
gitignore_content = """.env
data/
models/*.pkl
models/*.npy
__pycache__/
.ipynb_checkpoints/
*.pyc
.DS_Store
"""

with open('.gitignore', 'w') as f:
    f.write(gitignore_content)

print("Created .gitignore")

In [ ]:
import os
print(os.path.exists('.gitignore'))
with open('.gitignore', 'r') as f:
    print(f.read())
    

In [ ]:
import os
print(os.getcwd())  # should show Credit_Risk_Ai path
print(os.path.exists('.gitignore'))  # should be True

In [ ]:
import os
print(os.path.exists('requirements.txt'))

In [ ]:
readme_content = '''# Smart Loan Risk Assessment — ML + Generative AI Explainability

A credit risk prediction system that combines a traditional machine learning
model with a large language model (Gemini) to produce clear, human-readable
explanations for every prediction — tailored separately for loan officers and
for applicants.

> **Portfolio / educational project.** This is not a real financial decision
> system. See the [Responsible AI](#responsible-ai--fairness-considerations)
> section below for important caveats around bias and fairness.

---

## Why this project

Most credit risk models output a single number: a probability of default.
That number is not very useful on its own — a loan officer needs to know
*why* the model reached that conclusion, and an applicant deserves a plain-English
answer rather than a rejection with no explanation.

This project pairs a classic ML pipeline with two explainability layers:

1. **SHAP** — quantifies exactly which features pushed an individual
   applicant's risk score up or down.
2. **Gemini (LLM)** — translates those raw SHAP values into natural,
   audience-appropriate language, without technical jargon.

---

## Architecture

```
                    ┌─────────────────────┐
   Applicant data → │  Random Forest       │ → Risk prediction (0–1)
                    │  Classifier          │
                    └─────────┬────────────┘
                              │
                              ▼
                    ┌─────────────────────┐
                    │  SHAP TreeExplainer  │ → Top contributing features
                    │  (per-applicant)     │    + direction of impact
                    └─────────┬────────────┘
                              │
                              ▼
                    ┌─────────────────────┐
                    │  Gemini 2.5 Flash    │ → Natural-language explanation
                    │  (prompted, with     │    (loan officer / applicant
                    │   fairness rules)    │     tone)
                    └─────────────────────┘
```

---

## Dataset

[Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk)
(Kaggle competition dataset), ~307,000 loan applications with 122 features
covering demographics, income, employment, and existing credit history.

- **Target:** binary — did the applicant default on the loan (1) or repay it (0)?
- **Class balance:** ~92% repaid / ~8% defaulted — a realistic, heavily
  imbalanced classification problem.

The raw CSVs are not included in this repository (see `.gitignore`) due to
size. To reproduce this project, download the dataset from Kaggle and place
`application_train.csv` in a `data/` folder at the project root.

---

## Modeling approach

| Model | ROC-AUC |
|---|---|
| Decision Tree (baseline, `max_depth=6`) | 0.717 |
| **Random Forest** (`n_estimators=200`, `max_depth=10`) | **0.733** |

Both models used `class_weight='balanced'` to compensate for the ~8% minority
(default) class. Median imputation was used for missing numeric values, and
categorical variables were one-hot encoded.

**Classification report (Random Forest, threshold = 0.5):**

| Class | Precision | Recall | F1 |
|---|---|---|---|
| Repaid (0) | 0.96 | 0.71 | 0.81 |
| Default (1) | 0.16 | 0.64 | 0.26 |

The model prioritizes **recall on defaulters over precision** — a deliberate
effect of class balancing. In a lending context this reflects a real business
tradeoff: missing an actual defaulter (false negative) is typically costlier
than incorrectly flagging a safe applicant for review (false positive). This
threshold could be tuned further depending on an institution's risk appetite.

**Top predictive features** (Random Forest importances, confirmed independently
by SHAP): `EXT_SOURCE_1/2/3` (external credit bureau scores), applicant age,
length of employment, and loan/goods value.

---

## Explainability layer (SHAP)

For every applicant, `shap.TreeExplainer` produces per-feature contribution
values, which are converted into a simple structured format:

```python
{'feature': 'EXT_SOURCE_2', 'value': 0.594, 'effect': 'decreases risk', 'impact_score': 0.0338}
{'feature': 'DAYS_EMPLOYED', 'value': -105, 'effect': 'increases risk', 'impact_score': 0.0143}
```

This structured output is what gets passed to the LLM in the next step — the
model never sees raw applicant data, only the already-computed, ranked
contributing factors.

---

## AI-generated explanations (Gemini)

The top 5 SHAP factors for an applicant are converted into a prompt for
**Gemini 2.5 Flash**, which generates a short, plain-language explanation.
The same underlying data produces two different outputs depending on the
intended audience:

**Loan officer view** (precise, analytical):
> The model classifies this applicant as 'Low Risk' for default, predicting a
> 45.80% probability. This assessment is largely due to several strong
> positive factors, including a robust external credit indicator and a
> favorable education profile...

**Applicant view** (empathetic, constructive — high-risk example):
> Our review of your application suggests a higher likelihood of default,
> with the model estimating this at 73.14%. A significant factor contributing
> to this assessment is the information gathered from external credit
> sources... We encourage you to review your credit report and explore steps
> to strengthen your overall credit profile for future applications.

---

## Responsible AI / fairness considerations

This is the part of the project I'd consider most important, and the reason
I'd encourage anyone building something similar to look past the accuracy
number.

**What we found:** Early testing showed the model's SHAP explanations
sometimes surfaced **age**, **gender**, and **geographic/regional** features
among the top contributing factors for a prediction — and, before a
safeguard was added, an early version of the AI explanation repeated this
directly to a simulated applicant (e.g., citing their age as a reason for
a high-risk classification).

**Why this matters:** In many jurisdictions, using age, gender, or
neighborhood/region as an explicit factor in a credit decision — or
disclosing it as a reason to the applicant — raises fair-lending and
anti-discrimination concerns (in the US, for example, under the Equal Credit
Opportunity Act). Geographic risk scoring in particular closely resembles
historical **redlining** practices.

**What was done about it:** The Gemini prompt was updated with an explicit
constraint instructing the model to never cite age, gender, education level,
or region as a justification, and to instead focus only on financial and
credit-history factors — while transparently noting that other (undisclosed)
factors were also considered. The before/after difference is a concrete,
testable example of prompt-level fairness mitigation in this repository's
notebook.

**What a production system would still need:** removing or auditing these
features at the model level (not just at the explanation layer), a formal
disparate-impact analysis, and legal/compliance review before any real-world
use. This project treats that as future work, not a solved problem.

---

## Interactive demo (Streamlit)

`app.py` provides a simple UI to explore the pipeline:

- Select any sample applicant from the test set
- View their risk classification and top SHAP-driving factors
- Toggle between "Loan Officer" and "Applicant" explanation styles
- See the live Gemini-generated explanation

Run locally:

```bash
conda activate creditrisk   # or your preferred environment
pip install -r requirements.txt
streamlit run app.py
```

You will need your own Gemini API key (free from
[Google AI Studio](https://aistudio.google.com/)) in a `.env` file:

```
GOOGLE_API_KEY=your_key_here
```

---

## Tech stack

- **Python**, pandas, NumPy, scikit-learn (Decision Tree, Random Forest)
- **SHAP** for model explainability
- **Google Gen AI SDK** (`google-genai`) with Gemini 2.5 Flash for natural
  language generation
- **Streamlit** for the interactive demo
- **Matplotlib / Seaborn** for EDA and visualizations

---

## Project structure

```
Credit_Risk_Ai/
├── HomeRisk.ipynb          # EDA, model training, SHAP, Gemini prototyping
├── app.py                  # Streamlit demo app
├── requirements.txt
├── .env                    # API key (not committed)
├── .gitignore
├── data/                   # Home Credit dataset (not committed, see above)
└── models/                 # Saved model + SHAP artifacts (not committed)
```

---

## Possible next steps

- Enrich the model with aggregated features from `bureau.csv` (applicants'
  credit history at other institutions) and compare AUC uplift
- Threshold tuning / cost-sensitive evaluation for the precision-recall
  tradeoff described above
- Formal fairness audit (e.g., disparate impact ratio across demographic
  groups) rather than the prompt-level mitigation used here
- Deploy the Streamlit app publicly (Streamlit Community Cloud) with request
  caching to manage LLM API rate limits

---

## Disclaimer

This project uses a public Kaggle dataset for educational purposes only. It
is not intended for and should not be used to make real lending or credit
decisions.'''

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("README.md created")

In [ ]:
readme_content = r'''# Smart Loan Risk Assessment — ML + Generative AI Explainability

A credit risk prediction system that combines a traditional machine learning
model with a large language model (Gemini) to produce clear, human-readable
explanations for every prediction — tailored separately for loan officers and
for applicants.

> **Portfolio / educational project.** This is not a real financial decision
> system. See the [Responsible AI](#responsible-ai--fairness-considerations)
> section below for important caveats around bias and fairness.
'''

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("README.md created")

In [ ]:
import os
print(os.listdir())

In [ ]:
import os
import shutil

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Move CSV files into data/
csv_files = ['application_test.csv', 'application_train.csv', 'bureau.csv',
             'bureau_balance.csv', 'credit_card_balance.csv',
             'HomeCredit_columns_description.csv', 'installments_payments.csv',
             'POS_CASH_balance.csv', 'previous_application.csv']

for f in csv_files:
    if os.path.exists(f):
        shutil.move(f, os.path.join('data', f))
        print(f"Moved {f} -> data/")

# Move model artifact files into models/ (only if not already there)
model_files = ['credit_risk_rf.pkl', 'feature_columns.pkl', 'imputer.pkl',
               'shap_values_class1.npy', 'X_test_sample.pkl']

for f in model_files:
    if os.path.exists(f):
        shutil.move(f, os.path.join('models', f))
        print(f"Moved {f} -> models/")

print("\nDone. Root folder now contains:")
print(os.listdir())

In [ ]:
print("Root:", os.listdir())
print("\ndata/:", os.listdir('data'))
print("\nmodels/:", os.listdir('models'))

In [ ]:
import os
import shutil

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

csv_files = ['application_test.csv', 'application_train.csv', 'bureau.csv',
             'bureau_balance.csv', 'credit_card_balance.csv',
             'HomeCredit_columns_description.csv', 'installments_payments.csv',
             'POS_CASH_balance.csv', 'previous_application.csv']

for f in csv_files:
    if os.path.exists(f):
        shutil.move(f, os.path.join('data', f))
        print(f"Moved {f} -> data/")

model_files = ['credit_risk_rf.pkl', 'feature_columns.pkl', 'imputer.pkl',
               'shap_values_class1.npy', 'X_test_sample.pkl']

for f in model_files:
    if os.path.exists(f):
        shutil.move(f, os.path.join('models', f))
        print(f"Moved {f} -> models/")

print("\nDone. Root folder now contains:")
print(os.listdir())

In [ ]:
print("Root:", os.listdir())
print("data/:", os.listdir('data'))
print("models/:", os.listdir('models'))

In [ ]:
import os
os.makedirs('screenshots', exist_ok=True)
print(os.path.exists('screenshots'))

In [ ]:
print(os.listdir('screenshots'))

In [ ]:
with open('HomeRisk.ipynb', 'r', encoding='utf-8') as f:
    content = f.read()
print('AIza' in content)  # Gemini/Google API keys typically start with "AIza"

In [ ]:
with open('HomeRisk.ipynb', 'r', encoding='utf-8') as f:
    content = f.read()

print("Contains 'AIza':", 'AIza' in content)
print("Contains 'GOOGLE_API_KEY':", 'GOOGLE_API_KEY' in content)

# Find and print some context around any match, so we can see exactly which cell it's in
import re
for match in re.finditer(r'.{50}AIza.{50}', content):
    print(match.group())
    print('---')

In [1]:
with open('HomeRisk.ipynb', 'r', encoding='utf-8') as f:
    content = f.read()

print("Contains 'AIza':", 'AIza' in content)
print("Contains 'AQ.':", 'AQ.' in content)
print("Contains 'GOOGLE_API_KEY':", 'GOOGLE_API_KEY' in content)

Contains 'AIza': True
Contains 'AQ.': False
Contains 'GOOGLE_API_KEY': True


In [2]:
import re

with open('HomeRisk.ipynb', 'r', encoding='utf-8') as f:
    content = f.read()

print("=== AIza matches ===")
for match in re.finditer(r'.{80}AIza.{80}', content):
    print(match.group())
    print('---')

print("\n=== GOOGLE_API_KEY matches ===")
for match in re.finditer(r'.{80}GOOGLE_API_KEY.{80}', content):
    print(match.group())
    print('---')

=== AIza matches ===

=== GOOGLE_API_KEY matches ===


In [3]:
import re

with open('HomeRisk.ipynb', 'r', encoding='utf-8') as f:
    content = f.read()

# Real Google API keys: AIza followed by 35 alphanumeric/underscore/hyphen chars
matches_aiza = re.findall(r'AIza[0-9A-Za-z_\-]{35}', content)

# Newer format keys starting with AQ.
matches_aq = re.findall(r'AQ\.[A-Za-z0-9_\-\.]{20,}', content)

print("Real AIza-style keys found:", len(matches_aiza))
print("Real AQ.-style keys found:", len(matches_aq))

if matches_aiza:
    print("Sample:", matches_aiza[0][:10] + "...")
if matches_aq:
    print("Sample:", matches_aq[0][:10] + "...")

Real AIza-style keys found: 0
Real AQ.-style keys found: 0
